# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

RAW_DATA_DIR = PROJECT_ROOT / "data/raw"
VAR_EXPLANATIONS_PATH = PROJECT_ROOT / "data/_variable_explanations.csv"

In [ ]:
var_explanations = pd.read_csv(
    VAR_EXPLANATIONS_PATH,
    sep=";",
    engine="python",
    encoding="latin-1"
)

if {"Variable", "Description"}.issubset(var_explanations.columns):
    column_map = {row["Variable"]: f"{row['Variable']} ({row['Description']})" for _, row in var_explanations.iterrows()}
else:
    column_map = {}

var_explanations[["Variable", "Description"]].head(10)

In [ ]:
# read all Excel files from the folder into the dictionary of data frames
data_frames = {}
for file_path in RAW_DATA_DIR.glob("*.xlsx"):
    df = pd.read_excel(file_path)
    data_frames[file_path.stem] = df

In [ ]:
# # plot all time series in the first data frame each time series in separate figure. Add titles, subscribe axes

# import matplotlib.pyplot as plt

# for column in first_df.columns:
#     plt.figure()
#     plt.plot(first_df.index, first_df[column])
#     plt.title(column)
#     plt.xlabel("Index")
#     plt.ylabel("Value")
#     plt.grid()
#     plt.show()


In [ ]:
# extract the target values for prediction and store the dataframe and targets in csv file. The target values are the first rows of the corresponding columns: 
# - VO2peak_abs (Absolute peak oxygen uptake achieved during test)
# - VO2peak_rel (Peak oxygen uptake normalized/relative to body mass)
# - PeakPower (Maximum achieved work rate during test)

target_variables = ["VO2peak_abs", "VO2peak_rel", "PeakPower"]
processed_dir = PROJECT_ROOT / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)

for file_name, df in data_frames.items():
    print(f"Processing {file_name}...")
    resolved_targets = []
    for base_name in target_variables:
        descriptive_name = column_map.get(base_name, base_name)
        if descriptive_name in df.columns:
            resolved_targets.append((base_name, descriptive_name))
        elif base_name in df.columns:
            resolved_targets.append((base_name, base_name))
        else:
            print(f"Warning: {base_name} not found in {file_name}; skipping this target.")
    target_column_labels = [label for _, label in resolved_targets]

    if not target_column_labels:
        print(f"No target columns found in {file_name}; skipping export.")
        continue

    targets = df[target_column_labels].iloc[0].rename({label: base for base, label in resolved_targets})
    data_without_targets = df.drop(columns=target_column_labels, errors="ignore")

    output_data_path = processed_dir / f"{file_name}_data.csv"
    output_targets_path = processed_dir / f"{file_name}_targets.csv"
    data_without_targets.to_csv(output_data_path, index=False)
    targets.to_frame().T.to_csv(output_targets_path, index=False)




In [ ]:
# plot the column "VO2pulm" of all subjects in a single figure
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
for file_name, df in data_frames.items():
    # if "VO2pulm" in df.columns:
    #     plt.plot(df["VO2pulm"], label=file_name)
    if "WorkRate" in df.columns:
        plt.plot(df["WorkRate"], label=f"{file_name} - WorkRate", linestyle='--')
plt.title("Work Rate Time Series for All Subjects")
plt.xlabel("Time Index")
plt.ylabel("Work Rate")
plt.legend()
plt.grid()

In [ ]:
# plot columns "WorkRate" and "VO2pulm" of first three subjects in separate figures, 
for i, (file_name, df) in enumerate(data_frames.items()):
    if i >= 3:
        break
    plt.figure(figsize=(12, 6))
    if "WorkRate" in df.columns:
        plt.plot(df["WorkRate"], label="WorkRate")
    # if "VO2pulm" in df.columns:
    #     plt.plot(df["VO2pulm"], label="VO2pulm")
    plt.title(f"Work Rate and VO2pulm for {file_name}")
    plt.xlabel("Time Index")
    plt.ylabel("Value")
    plt.legend()
    plt.grid()